# FCVG101 — Tìm sản phẩm thời trang bằng ảnh

Bài toán: cho một ảnh sản phẩm (query), tìm trong kho ảnh của shop (gallery) những ảnh của **đúng món đó** (cùng `item_id`).
Dataset: DeepFashion In-shop. Món ở tập test **không có** trong tập train, nên đây là bài toán *tìm kiếm theo độ giống*, không phải phân loại.

Hướng làm: dùng CLIP ViT-B/32 đóng băng làm baseline, rồi thêm một mạng nhỏ (adapter) học trên vector CLIP.
Mọi con số trong notebook do code tính và in ra; chạy từ trên xuống.

| Phần | Nội dung |
|---|---|
| 1 | Dữ liệu, chia train / val / test |
| 2 | Tiền xử lý và augmentation |
| 3 | Baseline: CLIP đóng băng, histogram màu |
| 4 | Adapter + ArcFace |
| 5 | So sánh các cấu hình trên validation |
| 6 | Test (chạy một lần) |
| 7 | Demo, phân tích lỗi, kết luận |


In [ ]:
# Colab: !pip install -q ftfy regex git+https://github.com/openai/CLIP.git
import copy, hashlib, math, time
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import clip
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageDraw
from torchvision import transforms as T

# thư mục chứa img/, Eval/, list_bbox_inshop.txt (tự tìm ngược lên các thư mục cha)
DATA = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Eval/list_eval_partition.txt").exists()), None)
# Colab: DATA = Path("/content/drive/MyDrive/FCVG101_data")
assert DATA, "Không thấy dataset: đặt DATA = thư mục chứa img/, Eval/, list_bbox_inshop.txt"
CACHE = Path("cache/embeddings"); CACHE.mkdir(parents=True, exist_ok=True)   # vector CLIP lưu lại để khỏi mã hóa lại
device = "cuda" if torch.cuda.is_available() else "cpu"

clip_model, clip_pre = clip.load("ViT-B/32", device=device, jit=False, download_root=".cache/clip")
clip_model.eval()
for p in clip_model.parameters():
    p.requires_grad_(False)          # CLIP đóng băng
print(device, "| CLIP tham số:", sum(p.numel() for p in clip_model.parameters()) // 10**6, "triệu | dữ liệu:", DATA)


## 1. Dữ liệu


In [ ]:
# file nhãn: dòng 1 là số dòng, dòng 2 là tiêu đề
rows = lambda f: [l.split() for l in (DATA / f).read_text().splitlines()[2:] if l.strip()]
part = rows("Eval/list_eval_partition.txt")                       # path, item_id, split
bbox_rows = rows("list_bbox_inshop.txt")                          # path, clothes_type, pose_type, x1, y1, x2, y2
box = {r[0]: tuple(map(int, r[3:7])) for r in bbox_rows}          # bbox trên ảnh 256x256
POSES = {"1": "trước", "2": "nghiêng", "3": "sau", "4": "toàn thân", "5": "cận cảnh", "6": "chỉ sản phẩm"}
pose = {r[0]: POSES[r[2]] for r in bbox_rows}
category = lambda p: "/".join(p.split("/")[1:3])                  # vd WOMEN/Dresses

by_split = {s: [(p, i) for p, i, sp in part if sp == s] for s in ("train", "query", "gallery")}
print(f"{len(part):,} ảnh, {len({r[1] for r in part}):,} món, {len({category(r[0]) for r in part})} nhóm sản phẩm")
for s, v in by_split.items():
    print(f"  {s:8s} {len(v):6,} ảnh  {len({i for _, i in v}):5,} món")

# phân bố nhóm sản phẩm theo split
cats = sorted({category(p) for p, _ in by_split["train"]})
fig, ax = plt.subplots(figsize=(10, 4.2))
w = 0.27
for k, s in enumerate(by_split):
    n = Counter(category(p) for p, _ in by_split[s])
    ax.bar(np.arange(len(cats)) + (k - 1) * w, [n[c] for c in cats], w, label=s)
ax.set_xticks(np.arange(len(cats)), cats, rotation=75, ha="right", fontsize=7); ax.set_ylabel("số ảnh"); ax.legend()
ax.set_title("Số ảnh theo nhóm sản phẩm"); plt.tight_layout(); plt.show()

# một sản phẩm, nhiều góc chụp
ex = defaultdict(list)
for p, i in by_split["train"]:
    ex[i].append(p)
ex_item = next(i for i, ps in ex.items() if len(ps) >= 4)
fig, axes = plt.subplots(1, 4, figsize=(10, 2.8))
for a, p in zip(axes, ex[ex_item][:4]):
    a.imshow(Image.open(DATA / p)); a.set_title(pose[p], fontsize=9); a.axis("off")
plt.suptitle(f"Cùng một món ({ex_item}), nhiều góc chụp"); plt.tight_layout(); plt.show()


**Chia dữ liệu.** Tập test chính thức giữ nguyên và không dùng để chọn gì. Để chọn cấu hình ta cần một tập validation:
lấy 10% **món** (không phải 10% ảnh) từ tập train, chia ảnh của món đó thành query và gallery.
Chia theo món để val cũng là "món chưa thấy" như test; chia theo ảnh thì mô hình đã thấy món đó lúc train và val sẽ dễ một cách giả tạo.


In [ ]:
def rank_key(seed, key):
    return hashlib.sha256(f"{seed}:{key}".encode()).hexdigest()

def split_val(train_rows, fraction=0.1, seed="fcvg101-val"):
    by_id = defaultdict(list)
    for p, i in train_rows:
        by_id[i].append(p)
    eligible = sorted((i for i, ps in by_id.items() if len(ps) >= 2), key=lambda i: rank_key(seed, i))
    val_ids = set(eligible[:round(len(by_id) * fraction)])
    tr, vq, vg = [], [], []
    for i, ps in by_id.items():
        if i not in val_ids:
            tr += [(p, i) for p in ps]
            continue
        ps = sorted(ps, key=lambda p: rank_key(seed, p))
        n = max(1, len(ps) // 2)
        vq += [(p, i) for p in ps[:n]]
        vg += [(p, i) for p in ps[n:]]
    return tr, vq, vg

tr, vq, vg = split_val(by_split["train"])
ids = lambda rows: {i for _, i in rows}
test_ids = ids(by_split["query"]) | ids(by_split["gallery"])
assert not ids(tr) & (ids(vq) | ids(vg)), "train và val trùng món"
assert not (ids(tr) | ids(vq) | ids(vg)) & test_ids, "train/val trùng món với test"
assert not {p for p, _ in by_split["query"]} & {p for p, _ in by_split["gallery"]}, "ảnh vừa query vừa gallery"
assert ids(vq) == ids(vg)
print(f"adapter-train {len(tr):,} ảnh / {len(ids(tr)):,} món | val {len(vq):,} query + {len(vg):,} gallery / {len(ids(vq))} món")
print(f"test {len(by_split['query']):,} query + {len(by_split['gallery']):,} gallery / {len(test_ids):,} món (giữ nguyên toàn bộ gallery)")


## 2. Tiền xử lý và augmentation

CLIP nhận ảnh vuông 224×224. Có hai cách chuẩn bị ảnh:
- **full**: dùng cả ảnh (resize + chuẩn hóa như CLIP gốc).
- **crop**: cắt theo bbox sản phẩm (nới 10%, cạnh tối thiểu 64 px), rồi **pad cho vuông** để CLIP không cắt mất mép; thử pad màu trung bình và pad trắng.

Augmentation chỉ dùng khi train: crop ngẫu nhiên 75–100%, lật ngang, đổi sáng/tương phản ±20%, xóa một mảng nhỏ.
Không đổi sắc độ vì màu là đặc điểm nhận dạng sản phẩm.


In [ ]:
MEAN_RGB = tuple(round(255 * m) for m in (0.48145466, 0.4578275, 0.40821073))

def crop_box(b, w, h, margin=0.1, min_side=64):
    bw, bh = (b[2] - b[0]) * (1 + 2 * margin), (b[3] - b[1]) * (1 + 2 * margin)
    bw, bh = max(bw, min(min_side, w)), max(bh, min(min_side, h))
    cx, cy = (b[0] + b[2]) / 2, (b[1] + b[3]) / 2
    x1 = min(max(cx - bw / 2, 0), w - bw)          # dịch (không co) box nếu tràn biên
    y1 = min(max(cy - bh / 2, 0), h - bh)
    x1, y1 = max(0, round(x1)), max(0, round(y1))
    return x1, y1, min(w, round(x1 + bw)), min(h, round(y1 + bh))

def prep(im, path, mode):
    # mode: full | mean (crop + pad màu trung bình) | white (crop + pad trắng)
    im = im.convert("RGB")
    if mode == "full":
        return im
    c = im.crop(crop_box(box[path], *im.size))
    side = max(c.size)
    canvas = Image.new("RGB", (side, side), (255, 255, 255) if mode == "white" else MEAN_RGB)
    canvas.paste(c, ((side - c.width) // 2, (side - c.height) // 2))
    return canvas

train_aug = T.Compose([T.RandomResizedCrop(224, scale=(0.75, 1.0), ratio=(0.9, 1.1),
                                           interpolation=T.InterpolationMode.BICUBIC),
                       T.RandomHorizontalFlip(), T.ColorJitter(brightness=0.2, contrast=0.2)])
erase = T.RandomErasing(p=0.25, scale=(0.02, 0.1), value=0)
aug_tf = lambda im: erase(clip_pre(train_aug(im)))        # PIL -> tensor, chỉ dùng cho train

# minh họa: lấy ảnh đầu tiên của mỗi kiểu chụp (trước / nghiêng / cận cảnh)
shown = [next(p for p, _ in by_split["train"] if pose[p] == k) for k in ("trước", "nghiêng", "cận cảnh")]
fig, axes = plt.subplots(len(shown), 4, figsize=(9, 2.4 * len(shown)))
for row, p in zip(axes, shown):
    im = Image.open(DATA / p).convert("RGB")
    marked = im.copy(); ImageDraw.Draw(marked).rectangle(box[p], outline=(255, 0, 0), width=2)
    for a, (img, title) in zip(row, [(marked, f"ảnh gốc + bbox ({pose[p]})"), (prep(im, p, "full"), "full"),
                                    (prep(im, p, "mean"), "crop + pad trung bình"), (prep(im, p, "white"), "crop + pad trắng")]):
        a.imshow(img); a.set_title(title, fontsize=8); a.axis("off")
plt.tight_layout(); plt.show()

p = shown[0]; im = Image.open(DATA / p).convert("RGB")
fig, axes = plt.subplots(1, 5, figsize=(11, 2.4))
axes[0].imshow(im); axes[0].set_title("gốc", fontsize=8)
for k, a in enumerate(axes[1:]):
    a.imshow(erase(T.ToTensor()(train_aug(im))).permute(1, 2, 0)); a.set_title(f"augment {k + 1}", fontsize=8)
for a in axes: a.axis("off")
plt.tight_layout(); plt.show()


**Vector đặc trưng.** CLIP đóng băng cho cùng một ảnh luôn ra cùng một vector, nên mã hóa mỗi ảnh một lần rồi lưu vào `cache/embeddings/`
(lần chạy sau đọc lại, không mã hóa lại). Lần đầu trên máy mới việc này mất từ vài phút đến hơn một giờ tùy tốc độ đọc ổ đĩa.


In [ ]:
def encode(paths, mode, aug=None, workers=8):
    # ảnh -> vector CLIP 512 chiều, chuẩn hóa độ dài 1
    def load(p):
        with Image.open(DATA / p) as im:
            x = prep(im, p, mode)
        return aug(x) if aug else clip_pre(x)
    out = []
    with ThreadPoolExecutor(workers) as ex, torch.inference_mode():
        for i in range(0, len(paths), 256):
            x = torch.stack(list(ex.map(load, paths[i:i + 256]))).to(device)
            out.append(F.normalize(clip_model.encode_image(x).float(), dim=-1).cpu())
    return torch.cat(out).numpy()

def feats(name, paths, mode):
    f, pf = CACHE / f"{name}.npy", CACHE / f"{name}.paths.txt"
    if f.exists() and pf.read_text().splitlines() == paths:
        return np.load(f)
    t = time.time(); x = encode(paths, mode)
    np.save(f, x); pf.write_text("\n".join(paths) + "\n")
    print(f"  mã hóa {name}: {len(paths):,} ảnh trong {time.time() - t:.0f}s")
    return x

all_train = [p for p, _ in by_split["train"]]
row = {p: k for k, p in enumerate(all_train)}
X_full = feats("train_full", all_train, "full")                      # (25882, 512)
X_white = feats("train_A1_white_pad", all_train, "white")
sel = lambda X, rs: X[[row[p] for p, _ in rs]]

q_paths = [p for p, _ in by_split["query"]]; g_paths = [p for p, _ in by_split["gallery"]]
item_idx = {i: k for k, i in enumerate(sorted({r[1] for r in part}))}      # item_id -> số nguyên
lab = lambda rs: np.array([item_idx[i] for _, i in rs])
print("vector train:", X_full.shape)


## 3. Baseline và cách chấm điểm

Tìm kiếm: cosine giữa vector query và toàn bộ vector gallery, xếp giảm dần.
**Recall@K** = tỉ lệ query có ảnh đúng món nằm trong K ảnh đầu; **MRR** = trung bình của 1/hạng của ảnh đúng đầu tiên.
Baseline (E0) là CLIP đóng băng, không học gì thêm. Để so sánh biểu diễn, thêm histogram màu HSV (biểu diễn thủ công).


In [ ]:
@torch.no_grad()
def retrieve(Q, G, qid, gid, k=5, bs=1024):
    # trả về hạng của ảnh đúng đầu tiên cho từng query, và top-k chỉ số gallery
    Q, G = torch.as_tensor(Q, device=device), torch.as_tensor(G, device=device)
    qid, gid = torch.as_tensor(qid, device=device), torch.as_tensor(gid, device=device)
    ranks, top = [], []
    for i in range(0, len(Q), bs):
        s = Q[i:i + bs] @ G.T                                                   # cosine (vector đã chuẩn hóa)
        best = s.masked_fill(gid[None] != qid[i:i + bs, None], -2).max(1).values  # điểm của ảnh đúng cao nhất
        ranks.append((s > best[:, None]).sum(1) + 1)
        top.append(s.topk(k, 1).indices)
    return torch.cat(ranks).cpu().numpy(), torch.cat(top).cpu().numpy()

def scores(r):
    return {"R@1": (r == 1).mean(), "R@5": (r <= 5).mean(), "R@10": (r <= 10).mean(), "MRR": (1 / r).mean()}

def hsv_hist(im, bins=(8, 4, 4)):
    hsv = np.asarray(im.convert("RGB").convert("HSV")).reshape(-1, 3)
    h, _ = np.histogramdd(hsv, bins=bins, range=((0, 256),) * 3)
    h = h.ravel().astype(np.float32)
    return h / max(np.linalg.norm(h), 1e-12)

def hsv_feats(rs):
    with ThreadPoolExecutor(8) as ex:
        return np.stack(list(ex.map(lambda p: hsv_hist(prep(Image.open(DATA / p), p, "mean")), [p for p, _ in rs])))

qv, gv = lab(vq), lab(vg)
val_base = {
    "HSV histogram (thủ công, crop)": scores(retrieve(hsv_feats(vq), hsv_feats(vg), qv, gv)[0]),
    "E0 CLIP đóng băng, ảnh full": scores(retrieve(sel(X_full, vq), sel(X_full, vg), qv, gv)[0]),
    "E0 CLIP đóng băng, crop + pad trắng": scores(retrieve(sel(X_white, vq), sel(X_white, vg), qv, gv)[0]),
}
print(f"VALIDATION ({len(vq):,} query / {len(vg):,} gallery / {len(ids(vq))} món)")
for k, v in val_base.items():
    print(f"  {k:38s} R@1 {v['R@1']:6.2%}  R@5 {v['R@5']:6.2%}  MRR {v['MRR']:.3f}")


## 4. Adapter + ArcFace

**Adapter** (theo CLIP-Adapter, Gao et al. 2021): một mạng nhỏ 512→128→512 đặt sau CLIP. Kết quả = 0,2 × (đầu ra adapter) + 0,8 × (vector gốc),
rồi chuẩn hóa lại. Phần "giữ 80% vector gốc" (residual) để adapter hỏng cũng không phá kiến thức sẵn có của CLIP; nếu tỉ lệ bằng 0 thì chính là CLIP gốc.

**ArcFace** (Deng et al. 2019) là hàm loss khi train: coi mỗi món train là một lớp, kéo vector về tâm lớp đúng và thêm một lề góc để các món tách nhau rõ hơn.
Món test chưa từng thấy nên lớp phân loại này **bỏ đi sau khi train**; giữ lại là cách biến đổi vector.

Chỉ adapter và lớp ArcFace được học; CLIP không đổi. Chọn epoch tốt nhất theo Recall@1 trên **validation**.


In [ ]:
class Adapter(nn.Module):
    def __init__(self, dim=512, reduction=4, ratio=0.2):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(dim, dim // reduction, bias=False), nn.ReLU(),
                                 nn.Linear(dim // reduction, dim, bias=False), nn.ReLU())
        self.ratio = ratio
    def forward(self, f):
        f = F.normalize(f, dim=-1)
        return F.normalize(self.ratio * self.net(f) + (1 - self.ratio) * f, dim=-1)

class ArcFace(nn.Module):
    def __init__(self, dim, n_classes, s=30.0, m=0.3):
        super().__init__()
        self.w = nn.Parameter(torch.randn(n_classes, dim) * 0.01)
        self.s, self.m = s, m
    def forward(self, z, y):
        cos = F.linear(F.normalize(z, dim=-1), F.normalize(self.w, dim=-1)).clamp(-1 + 1e-7, 1 - 1e-7)
        theta = torch.acos(cos.gather(1, y[:, None]))
        target = torch.where(theta + self.m < math.pi, torch.cos(theta + self.m), cos.gather(1, y[:, None]))
        return F.cross_entropy(self.s * cos.scatter(1, y[:, None], target), y)

@torch.no_grad()
def embed(model, X):
    model.eval()
    X = torch.as_tensor(X, device=device)
    return torch.cat([model(X[i:i + 4096]) for i in range(0, len(X), 4096)]).cpu().numpy()

def train(X, y, val, seed, epochs=40, ratio=0.2, bs=256, lr=1e-3, head_lr=1e-2):
    # X: (N,512) hoặc (N,V,512) khi có nhiều bản augmented (mỗi bước lấy ngẫu nhiên 1 bản/ảnh)
    # val: (Xq, Xg, qid, gid); trả về trạng thái tốt nhất theo R@1 val
    torch.manual_seed(seed); g = torch.Generator().manual_seed(seed)
    X, y = torch.as_tensor(X, device=device), torch.as_tensor(y, device=device)
    model, head = Adapter(ratio=ratio).to(device), ArcFace(X.shape[-1], int(y.max()) + 1).to(device)
    opt = torch.optim.AdamW([{"params": model.parameters(), "lr": lr}, {"params": head.parameters(), "lr": head_lr}],
                            weight_decay=1e-4)
    best, log = None, []
    for ep in range(1, epochs + 1):
        model.train(); perm = torch.randperm(len(X), generator=g).to(device); tot = 0.0
        for i in range(0, len(X), bs):
            idx = perm[i:i + bs]; xb = X[idx]
            if xb.ndim == 3:
                xb = xb[torch.arange(len(idx), device=device), torch.randint(xb.shape[1], (len(idx),), generator=g).to(device)]
            loss = head(model(xb), y[idx])
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
            tot += loss.item() * len(idx)
        m = scores(retrieve(embed(model, val[0]), embed(model, val[1]), val[2], val[3])[0])
        log.append({"epoch": ep, "loss": tot / len(X), **m})
        if best is None or m["R@1"] > best["val"]["R@1"]:
            best = {"epoch": ep, "val": m, "state": copy.deepcopy(model.state_dict())}
    return best, log

n_params = sum(p.numel() for p in Adapter().parameters())
print(f"adapter: {n_params:,} tham số (CLIP: {sum(p.numel() for p in clip_model.parameters()):,})")

# thử một lần: E1 (ảnh full), seed 0
y_tr = lab(tr)
best, log = train(sel(X_full, tr), y_tr, (sel(X_full, vq), sel(X_full, vg), qv, gv), seed=0)
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].plot([r["loss"] for r in log]); ax[0].set_title("loss train (ArcFace)"); ax[0].set_xlabel("epoch")
ax[1].plot([r["R@1"] for r in log]); ax[1].axhline(val_base["E0 CLIP đóng băng, ảnh full"]["R@1"], ls="--", c="gray", label="E0")
ax[1].set_title("R@1 trên validation"); ax[1].set_xlabel("epoch"); ax[1].legend()
plt.tight_layout(); plt.show()
print(f"epoch tốt nhất {best['epoch']}: val R@1 {best['val']['R@1']:.2%}")


## 5. So sánh các cấu hình trên validation

Mỗi lần đổi **một** yếu tố so với E1:

| Mã | Ảnh đưa vào CLIP | Train adapter |
|---|---|---|
| E0 | full | không (baseline) |
| E1 | full | có |
| E2 | full + 3 bản augmented / ảnh | có |
| E3 | crop + pad trắng | có |

Mỗi cấu hình chạy 3 seed để thấy mức dao động.


In [ ]:
aug_name = "train_aug_C0_full_aug_v3_s0"
tr_paths = [p for p, _ in tr]
f, pf = CACHE / f"{aug_name}.npy", CACHE / f"{aug_name}.paths.txt"
if f.exists() and pf.read_text().splitlines() == tr_paths:
    A = np.load(f)
else:                                                   # 3 bản augmented / ảnh (ngẫu nhiên, không lặp lại được từng bit)
    A = np.stack([encode(tr_paths, "full", aug=aug_tf) for _ in range(3)], 1)
    np.save(f, A); pf.write_text("\n".join(tr_paths) + "\n")
X2 = np.concatenate([sel(X_full, tr)[:, None], A], 1)    # (N, 4, 512): bản gốc + 3 bản augmented
cos_views = float((A[:2000, 0] * A[:2000, 1]).sum(1).mean())
print(f"cosine trung bình giữa hai bản augmented của cùng một ảnh: {cos_views:.3f}")

setups = {
    "E1": (sel(X_full, tr), (sel(X_full, vq), sel(X_full, vg))),
    "E2": (X2,              (sel(X_full, vq), sel(X_full, vg))),
    "E3": (sel(X_white, tr), (sel(X_white, vq), sel(X_white, vg))),
}
SEEDS = (0, 1, 2)
STATES = Path("cache/adapter/nb_runs.pt")   # adapter đã train (9 cái); xóa file này nếu muốn train lại từ đầu (~12 phút trên GPU)
if STATES.exists():
    runs = torch.load(STATES, map_location=device, weights_only=False)
    print("nạp adapter đã train từ", STATES)
else:
    runs = {}
    for name, (Xtr, (Vq, Vg)) in setups.items():
        for seed in SEEDS:
            t = time.time()
            runs[name, seed], _ = train(Xtr, y_tr, (Vq, Vg, qv, gv), seed)
            print(f"{name} seed {seed}: epoch tốt nhất {runs[name, seed]['epoch']:2d}, val R@1 {runs[name, seed]['val']['R@1']:.2%} ({time.time() - t:.0f}s)")
    STATES.parent.mkdir(parents=True, exist_ok=True); torch.save(runs, STATES)


In [ ]:
def summarize(vals):
    a = np.array(vals) * 100
    return f"{a.mean():5.2f}  [{a.min():.2f} - {a.max():.2f}]"

print(f"{'cấu hình':34s} {'R@1 val TB [min-max]':26s} R@5 val")
for k in ("E0 CLIP đóng băng, ảnh full", "E0 CLIP đóng băng, crop + pad trắng"):
    print(f"{k:34s} {val_base[k]['R@1'] * 100:5.2f}{'':21s} {val_base[k]['R@5'] * 100:5.2f}")
for name, label in (("E1", "E1 adapter, full"), ("E2", "E2 adapter, full + augmented"), ("E3", "E3 adapter, crop + pad trắng")):
    r = [runs[name, s]["val"] for s in SEEDS]
    print(f"{label:34s} {summarize([x['R@1'] for x in r]):26s} {np.mean([x['R@5'] for x in r]) * 100:5.2f}")


Cách đọc: chênh lệch giữa các seed cho biết mức nhiễu; chỉ nên kết luận khi hai cấu hình cách nhau rõ hơn mức đó.
Val chỉ có vài trăm món nên dễ hơn test; ở đây val dùng để **so sánh** cấu hình với nhau, không dùng để báo độ chính xác.


## 6. Test chính thức

Các cấu hình và số epoch đã chốt trên validation ở trên. Ô dưới chạy trên test **đúng một lần**:
14.218 query so với toàn bộ 12.612 ảnh gallery. Không tham số nào được chỉnh theo kết quả test.


In [ ]:
Qf, Gf = feats("query", q_paths, "full"), feats("gallery", g_paths, "full")
Qw, Gw = feats("query_A1_white_pad", q_paths, "white"), feats("gallery_A1_white_pad", g_paths, "white")
qt, gt = lab(by_split["query"]), lab(by_split["gallery"])

test_feats = {"E1": (Qf, Gf), "E2": (Qf, Gf), "E3": (Qw, Gw)}
test, ranks = {}, {}
test["E0 full"], ranks["E0 full"] = None, retrieve(Qf, Gf, qt, gt)[0]
test["E0 crop + pad trắng"], ranks["E0 crop + pad trắng"] = None, retrieve(Qw, Gw, qt, gt)[0]
for name, (Q0, G0) in test_feats.items():
    ranks[name] = []
    for seed in SEEDS:
        m = Adapter().to(device); m.load_state_dict(runs[name, seed]["state"])
        ranks[name].append(retrieve(embed(m, Q0), embed(m, G0), qt, gt)[0])

print(f"TEST ({len(qt):,} query / {len(gt):,} gallery / {len(set(qt))} món); adapter: trung bình 3 seed [min-max R@1]")
table = {}
for name, r in ranks.items():
    rs = r if isinstance(r, list) else [r]
    s = [scores(x) for x in rs]
    table[name] = {k: np.mean([x[k] for x in s]) for k in s[0]}
    rng = f"  [{min(x['R@1'] for x in s) * 100:.2f} - {max(x['R@1'] for x in s) * 100:.2f}]" if len(rs) > 1 else ""
    print(f"  {name:22s} R@1 {table[name]['R@1'] * 100:5.2f}{rng:18s} R@5 {table[name]['R@5'] * 100:5.2f}  R@10 {table[name]['R@10'] * 100:5.2f}  MRR {table[name]['MRR']:.3f}")


## 7. Demo, phân tích lỗi, kết luận

**Demo.** Ba query test chọn ngẫu nhiên (có seed), top-5 của E0 và E3. Món ở test chưa từng có khi train. Viền xanh = đúng món.


In [ ]:
m3 = Adapter().to(device); m3.load_state_dict(runs["E3", 0]["state"])
picks = np.random.default_rng(2026).choice(len(qt), 3, replace=False)
fig, axes = plt.subplots(2 * len(picks), 6, figsize=(12, 4.3 * len(picks)))
E0 = (Qf, Gf); E3 = (embed(m3, Qw), embed(m3, Gw))
for n, qi in enumerate(picks):
    qp, qid_ = by_split["query"][qi]
    for r, (label, (Q, G)) in enumerate((("E0 CLIP gốc", E0), ("E3 adapter", E3))):
        _, top = retrieve(Q[qi:qi + 1], G, qt[qi:qi + 1], gt, k=5)
        row = axes[2 * n + r]
        row[0].imshow(Image.open(DATA / qp)); row[0].set_title(f"query · {label}", fontsize=8)
        for a, j in zip(row[1:], top[0]):
            gp, gid_ = by_split["gallery"][j]; ok = gid_ == qid_
            a.imshow(Image.open(DATA / gp)); a.set_title("đúng" if ok else "sai", fontsize=8, color="green" if ok else "red")
            for sp in a.spines.values(): sp.set_edgecolor("green" if ok else "red"); sp.set_linewidth(2)
        for a in row: a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()


**Ảnh bất kỳ.** Ô dưới nhận đường dẫn tới một ảnh sản phẩm bất kỳ (kể cả ảnh tự chụp) và trả về top-5 trong gallery.
Dùng cấu hình E1 (ảnh full) vì không cần bbox. Để trống `MY_IMAGE` thì dùng một ảnh query ngẫu nhiên.


In [ ]:
MY_IMAGE = None        # ví dụ: r"C:/anh/ao_cua_toi.jpg"
m1 = Adapter().to(device); m1.load_state_dict(runs["E1", 0]["state"])
G1 = embed(m1, Gf)
path = MY_IMAGE or q_paths[int(np.random.default_rng(7).integers(len(q_paths)))]
f_new = encode([str(path)], "full")                       # ảnh mới đi qua CLIP rồi adapter
sims = G1 @ embed(m1, f_new)[0]
top5 = np.argsort(-sims)[:5]
fig, axes = plt.subplots(1, 6, figsize=(12, 2.6))
axes[0].imshow(Image.open(DATA / path)); axes[0].set_title("ảnh đầu vào", fontsize=8)
for a, j in zip(axes[1:], top5):
    a.imshow(Image.open(DATA / by_split["gallery"][j][0])); a.set_title(f"cos {sims[j]:.3f}\n{by_split['gallery'][j][1]}", fontsize=7)
for a in axes: a.axis("off")
plt.tight_layout(); plt.show()


**Phân tích lỗi.** Không gắn nhãn thủ công: chia query theo thông tin có sẵn trong file nhãn (kiểu chụp, nhóm sản phẩm)
rồi xem E0 và E3 đúng bao nhiêu ở từng nhóm; và đếm query mà adapter sửa được hoặc làm hỏng.


In [ ]:
r0, r3 = ranks["E0 full"], ranks["E3"][0]            # E3 seed 0
ok0, ok3 = r0 == 1, r3 == 1
print(f"cả hai đúng ở hạng 1: {(ok0 & ok3).sum():6,}")
print(f"adapter sửa được    : {(~ok0 & ok3).sum():6,}   (E0 sai, E3 đúng)")
print(f"adapter làm hỏng    : {(ok0 & ~ok3).sum():6,}   (E0 đúng, E3 sai)")
print(f"cả hai sai          : {(~ok0 & ~ok3).sum():6,}")
fail = r3[~ok3]
print(f"khi E3 sai ở hạng 1, ảnh đúng nằm trong top-5 ở {(fail <= 5).mean():.0%} trường hợp, ngoài top-10 ở {(fail > 10).mean():.0%}")

def by_group(keys, min_n=30):
    groups = sorted({k for k in keys}, key=lambda g: ok0[[x == g for x in keys]].mean())
    groups = [g for g in groups if sum(1 for x in keys if x == g) >= min_n]
    a0 = [ok0[[x == g for x in keys]].mean() * 100 for g in groups]
    a3 = [ok3[[x == g for x in keys]].mean() * 100 for g in groups]
    return groups, a0, a3

fig, axes = plt.subplots(1, 2, figsize=(14, 6), gridspec_kw={"width_ratios": [1, 2]})
for ax, (keys, title) in zip(axes, ((np.array([pose[p] for p in q_paths]), "R@1 theo kiểu chụp của query"),
                                     (np.array([category(p) for p in q_paths]), "R@1 theo nhóm sản phẩm"))):
    gs, a0, a3 = by_group(keys)
    y = np.arange(len(gs))
    ax.barh(y - 0.2, a0, 0.4, label="E0", color="#9aa5b1"); ax.barh(y + 0.2, a3, 0.4, label="E3", color="#2f6fb0")
    ax.set_yticks(y, [f"{g} ({(keys == g).sum()})" for g in gs], fontsize=8); ax.set_title(title); ax.set_xlabel("R@1 (%)"); ax.legend()
plt.tight_layout(); plt.show()


In [ ]:
e0, e1, e2, e3 = (table[k]["R@1"] * 100 for k in ("E0 full", "E1", "E2", "E3"))
spread = max(max(scores(x)["R@1"] for x in ranks[k]) - min(scores(x)["R@1"] for x in ranks[k]) for k in ("E1", "E2", "E3")) * 100
print(f"Tóm tắt trên test: E0 {e0:.1f}% -> E1 {e1:.1f}% (+{e1 - e0:.1f} điểm), E2 {e2:.1f}%, E3 {e3:.1f}%.")
print(f"Dao động giữa 3 seed tối đa {spread:.1f} điểm, nên E1 và E2 khác nhau {abs(e1 - e2):.1f} điểm chưa đủ để nói cấu hình nào tốt hơn."
      if abs(e1 - e2) < spread * 2 else f"E1 và E2 khác nhau {abs(e1 - e2):.1f} điểm, lớn hơn mức dao động giữa seed ({spread:.1f}).")
print(f"Val so với test (E1): {np.mean([runs['E1', s]['val']['R@1'] for s in SEEDS]) * 100:.1f}% so với {e1:.1f}% - val dễ hơn vì ít món gây nhầm hơn.")


### Kết luận và giới hạn

- **Phần tái sử dụng:** CLIP ViT-B/32 (OpenAI), cấu trúc adapter của CLIP-Adapter, hàm loss ArcFace. **Phần tự làm:** pipeline dữ liệu và chia val theo món,
  so sánh cách chuẩn bị ảnh (full / crop / pad / augmentation), chuyển adapter sang bài toán tìm kiếm với ArcFace, đánh giá và phân tích lỗi.
  Đây không phải phương pháp mới.
- **Adapter giúp rõ rệt** so với CLIP đóng băng vì nó dạy vector phân biệt các món gần giống nhau, điều CLIP gốc không được học.
- **Augmentation gần như không đổi kết quả** vì CLIP đóng băng cho vector gần như giống nhau trước và sau augmentation (cosine ở phần 5),
  nên adapter không nhận thêm thông tin. Augmentation thường chỉ có ích khi chính encoder được học.
- **Crop có hai mặt:** làm CLIP đóng băng kém đi, nhưng khi đã có adapter thì không còn gây hại (xem bảng test).
- **Giới hạn:** các phương pháp fine-tune toàn bộ mạng trên cùng benchmark đạt cao hơn nhiều; adapter chỉ sắp xếp lại thông tin CLIP đã thấy.
  Cấu hình crop cần bbox, nên với ảnh ngoài dataset phải có thêm bước phát hiện sản phẩm. Chỉ dùng ảnh, chưa dùng mô tả văn bản.
- **Hướng tiếp:** fine-tune vài block cuối của CLIP, backbone mạnh hơn, xếp hạng lại top-10.

Tài liệu: Liu et al. DeepFashion (CVPR 2016); Radford et al. CLIP (ICML 2021); Gao et al. CLIP-Adapter (arXiv:2110.04544); Deng et al. ArcFace (CVPR 2019).
